In [ ]:
# repo root on sys.path, so `import segres` works when this notebook is
# run from scripts/ (its own directory) and not only from the repo root
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "scripts" else os.path.abspath(os.getcwd()))


In [ ]:
import os
import numpy as np
from PIL import Image
from segres.decode import load_annotations, build_lookup_tables, build_combined_mask


In [ ]:
output_dir = '../data/processed/mask'
json_path = '../data/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'

In [10]:
os.makedirs(output_dir, exist_ok=True)
 
data = load_annotations(json_path)
images_by_id, anns_by_image_id = build_lookup_tables(data)

total = len(images_by_id)
saved_count = 0
skipped_count = 0


In [11]:

for image_id, image_info in images_by_id.items():
    height = image_info["height"]
    width = image_info["width"]
    file_name = image_info["file_name"]

    annotations = anns_by_image_id.get(image_id, [])

    if len(annotations) == 0:
        skipped_count += 1
        continue

    mask = build_combined_mask(annotations, height, width)

    # save mask with same base name as image, but as PNG
    base_name = os.path.splitext(file_name)[0]
    mask_path = os.path.join(output_dir, f"{base_name}.png")

    # mask values are 0 or 1, scale to 0 or 255 so it's viewable as an image too
    mask_img = Image.fromarray((mask * 255).astype(np.uint8))
    mask_img.save(mask_path)

    saved_count += 1

print(f"Done. {saved_count} masks saved, {skipped_count} images skipped (no annotations).")
print(f"Total images processed: {total}")

Done. 1154 masks saved, 0 images skipped (no annotations).
Total images processed: 1154
